In [4]:
from __future__ import annotations

import hashlib
from typing import Sequence
import json

def sha256_hex(data: str) -> str:
    return hashlib.sha256(data.encode("utf-8")).hexdigest()

def sha256_concat(left: str, right: str) -> str:
    """SHA256 of the concatenation of two hex-digest strings, returning
    a new hex digest. This is the pairing step used at every level of
    the tree.
    """
    return hashlib.sha256((left + right).encode("utf-8")).hexdigest()


def merkle_root(leaf_hashes: Sequence[str]) -> str:
    """Compute the Merkle root of a list of transaction hashes.

    Using the same method as Bitcoin where we have a level with an odd number of nodes, the last
    node is paired with a duplicate of itself, rather than left unpaired.
    This makes sure that every level's pairing rule is identical and avoids the hassle of having to use a special
    case for "the odd one out."
    """
    if not leaf_hashes:
        raise ValueError("merkle_root() requires at least one leaf hash")

    level = list(leaf_hashes)
    while len(level) > 1:
        if len(level) % 2 == 1:
            level.append(level[-1])  # duplicate the odd node out
        level = [sha256_concat(level[i], level[i + 1]) for i in range(0, len(level), 2)]
    return level[0]


def merkle_proof(leaf_hashes: Sequence[str], index: int) -> list[tuple[str, str]]:
    """Return the Merkle proof (sibling path) for the leaf at `index`.

    Each of our enties is (sibling_hash, side), where the sides are denoted as "L" if the sibling
    is located on the left of our running hash at that level, or "R" if it
    sits to the right. This is exactly what an SPV client needs, in
    order, to recompute the root from a single leaf.
    """
    if not (0 <= index < len(leaf_hashes)):
        raise IndexError("index out of range for leaf_hashes")

    level = list(leaf_hashes)
    idx = index
    proof: list[tuple[str, str]] = []

    while len(level) > 1:
        if len(level) % 2 == 1:
            level.append(level[-1])

        if idx % 2 == 0:
            sibling_idx = idx + 1
            side = "R"  # sibling is to the right of our node
        else:
            sibling_idx = idx - 1
            side = "L"  # sibling is to the left of our node

        proof.append((level[sibling_idx], side))

        level = [sha256_concat(level[i], level[i + 1]) for i in range(0, len(level), 2)]
        idx //= 2

    return proof
    
def hash_transaction(tx: dict[str, Any]) -> str:
    """Canonical leaf hash: SHA256 over a stable JSON serialisation of the
    transaction (sorted keys, no whitespace) so the same format text always
    hashes the same way.
    """
    blob = json.dumps(tx, sort_keys=True, separators=(",", ":"))
    return sha256_hex(blob)
 

def verify_merkle_proof(leaf_hash: str, proof: list[tuple[str, str]], root: str) -> bool:
    """Recompute the root from `leaf_hash` and its proof; compare to `root`.

    This is the SPV check
    """
    current = leaf_hash
    for sibling_hash, side in proof:
        if side == "R":
            current = sha256_concat(current, sibling_hash)
        else:  # side == "L"
            current = sha256_concat(sibling_hash, current)
    return current == root


In [7]:
TRANSACTIONS: list[dict[str, Any]] = [
    {"sender": "Phenyo", "recipient": "Thato", "amount": 10.0, "timestamp": 1700000001},
    {"sender": "Thato", "recipient": "Phenyo", "amount": 4.0, "timestamp": 1700000002},
    {"sender": "Phenyo", "recipient": "Thato", "amount": 1.0, "timestamp": 1700000003},
    {"sender": "Thato", "recipient": "Phenyo", "amount": 7.5, "timestamp": 1700000004},
]
 
 
def main() -> None:
    print("*Transactions (the four leaves)*")
    for i, tx in enumerate(TRANSACTIONS, start=1):
        print(f"Tx{i}: {tx}")
 
    print("\n-*Leaf hashes (SHA256 of each canonical transaction)*-")
    leaf_hashes = [hash_transaction(tx) for tx in TRANSACTIONS]
    for i, h in enumerate(leaf_hashes, start=1):
        print(f"H{i} = {h}")
 
    print("\n-* Level-by-level pairing (hand-calculation trace)*-")
    h1, h2, h3, h4 = leaf_hashes
    h12 = sha256_concat(h1, h2)
    h34 = sha256_concat(h3, h4)
    root_hand = sha256_concat(h12, h34)
    print(f"H12 = SHA256(H1 + H2) = {h12}")
    print(f"H34 = SHA256(H3 + H4) = {h34}")
    print(f"ROOT = SHA256(H12 + H34) = {root_hand}")
 
    root_fn = merkle_root(leaf_hashes)
    print(f"\nmerkle_root(LEAF_HASHES) = {root_fn}")
    assert root_fn == root_hand
    print("Hand calculation matches merkle_root() output.")
 
    print("\n-* SPV proof for H2 (index 1)*-")
    index = 1
    proof = merkle_proof(leaf_hashes, index)
    print("Proof (sibling, side) pairs:", proof)
 
    ok = verify_merkle_proof(leaf_hashes[index], proof, root_fn)
    print(f"verify_merkle_proof(H2, proof, root) -> {ok}")
    assert ok
 
    tampered_tx2 = {**TRANSACTIONS[index], "amount": 999.0}
    tampered_hash = hash_transaction(tampered_tx2)
    ok_tampered = verify_merkle_proof(tampered_hash, proof, root_fn)
    print(f"verify_merkle_proof(tampered Tx2, proof, root) -> {ok_tampered}")
    assert ok_tampered is False
 
    print("\nAll Merkle / SPV checkpoints passed.")
 
 
if __name__ == "__main__":
    main()
 

*Transactions (the four leaves)*
Tx1: {'sender': 'Phenyo', 'recipient': 'Thato', 'amount': 10.0, 'timestamp': 1700000001}
Tx2: {'sender': 'Thato', 'recipient': 'Phenyo', 'amount': 4.0, 'timestamp': 1700000002}
Tx3: {'sender': 'Phenyo', 'recipient': 'Thato', 'amount': 1.0, 'timestamp': 1700000003}
Tx4: {'sender': 'Thato', 'recipient': 'Phenyo', 'amount': 7.5, 'timestamp': 1700000004}

-*Leaf hashes (SHA256 of each canonical transaction)*-
H1 = 5d29c42630357127ecdb665a8877686b35726099a78bd79f96740a17a80b2b87
H2 = 542ac27d01549c0c78a8269ebc8de1868d7c515dbc780b19ee441fac89278a64
H3 = cb7eab88ca0eb76bd91f1d22db4391462d6710a6d828fa573cfc6ad2ded73f7f
H4 = 8a96f30bea7d9b98fd31a915e38cf855390c357220f5308addb6345a87e421b5

-* Level-by-level pairing (hand-calculation trace)*-
H12 = SHA256(H1 + H2) = b51b10eac6388da9d98d14b0233bf28d29b478d787a1c2953b70f9bef2f4e7c2
H34 = SHA256(H3 + H4) = 522aaeee8f36d75128f3785b97082161d399541d298be0720e0be8d2377d15ec
ROOT = SHA256(H12 + H34) = 1a903b25428084d69eb